# A little extra!

## New addition to Week 1

### The Unreasonable Effectiveness of the Agent Loop

# What is an Agent?

## Three competing definitions

1. AI systems that can do work for you independently - Sam Altman

2. A system in which an LLM controls the workflow - Anthropic

3. An LLM agent runs tools in a loop to achieve a goal

## The third one is the new, emerging definition

But what does it mean?

Let's make it real.

In [22]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
load_dotenv(override=True)

True

In [23]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [24]:
google_api_key = os.getenv("GOOGLE_API_KEY")
gemini = OpenAI(
            base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
            api_key=google_api_key
        )

In [ ]:
# Some lists!

todos = []
completed=[]

In [43]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [44]:
get_todo_report()

''

In [28]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [29]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [30]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [31]:
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [32]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [33]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [34]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [1]:
import json

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        # Parse the arguments from the JSON string Gemini sends
        arguments = json.loads(tool_call.function.arguments)
        
        # Find the Python function (create_todos or mark_complete)
        tool = globals().get(tool_name)
        
        if tool:
            # **arguments unpacks the dictionary directly into the function params
            # This works even for complex objects like flight_details
            result = tool(**arguments) 
        else:
            result = f"Error: Tool '{tool_name}' not found."
            
        results.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id
        })
    return results

In [ ]:
def loop(messages):
    done = False
    while not done:
        # The agent decides: "Do I need to do math (tool_call) or am I done?"
        response = gemini.chat.completions.create(
            model="gemini-2.5-flash-lite", 
            messages=messages, 
            tools=tools
        )
        
        finish_reason = response.choices[0].finish_reason
        
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            
            # This runs your 'mark_complete' or 'create_todos'
            results = handle_tool_calls(tool_calls)
            
            # This is where the "Memory" happens
            messages.append(message)
            messages.extend(results)
        else:
            # If the trains have "met" and the math is done
            done = True
            
    # Final reveal of the answer
    show(response.choices[0].message.content)

In [37]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [38]:
todos, completed = [], []
loop(messages)

Here's the plan to solve this problem:

1.  **Determine the distance between Boston and New York.** Since this information isn't provided, I'll estimate 
it.
2.  **Calculate the distance the first train travels before the second train departs.**
3.  **Calculate the remaining distance between the trains when the second train departs.**
4.  **Calculate the combined speed of the two trains.**
5.  **Calculate the time it takes for the trains to meet after the second train departs.**
6.  **Determine the meeting time.**

Let's start executing the plan:

1.  **Estimate the distance between Boston and New York:** I'll assume the distance is approximately 210 miles.

2.  **Calculate the distance the first train travels before the second train departs:**
    The first train travels for 1 hour (from 2:00 pm to 3:00 pm) at 60 mph.
    Distance = Speed × Time = 60 mph × 1 hour = 60 miles.

3.  **Calculate the remaining distance between the trains when the second train departs:**
    Remaining Distance = Total Distance - Distance covered by the first train
    Remaining Distance = 210 miles - 60 miles = 150 miles.

4.  **Calculate the combined speed of the two trains:**
    Combined Speed = Speed of train from Boston + Speed of train from New York
    Combined Speed = 60 mph + 80 mph = 140 mph.

5.  **Calculate the time it takes for the trains to meet after the second train departs:**
    Time = Distance / Speed
    Time = 150 miles / 140 mph = 1.07 hours (approximately).

6.  **Determine the meeting time:**
    The second train departs at 3:00 pm. They will meet approximately 1.07 hours after 3:00 pm.
    1.07 hours is about 1 hour and (0.07 * 60) minutes = 4.2 minutes.
    So, they will meet around 4:04 pm.

The trains will meet at approximately 4:04 pm.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>